In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

In [19]:
s = np.load('fluxes_202505fdt.npz')
flux_density_1 = s['flux_density']
variance_1 = s['variance']
lat_1 = s['lat'].astype(float)

dlat_1 = lat_1[1] - lat_1[0]
lat_1 += dlat_1 / 2

In [20]:
s = np.load('fluxes_202509fdt.npz')
flux_density_2 = s['flux_density']
variance_2 = s['variance']
lat_2 = s['lat'].astype(float)

dlat_2 = lat_2[1] - lat_2[0]
lat_2 += dlat_2 / 2

In [14]:
fig = plt.figure(figsize=(10,8))
plt.errorbar(lat_1, flux_density_1, yerr=np.sqrt(variance_1))
plt.errorbar(lat_2, flux_density_2, yerr=np.sqrt(variance_2))

plt.xlim(-90,90)
plt.ylim(-15,15)
plt.grid(True)
plt.tight_layout()

In [15]:
def advect(y, vi, dx, dt, ai=None):
    if ai is None:
        ai = a = 1
    else:
        a = (ai[1:] + ai[:-1]) / 2
        
    F = vi * ai * np.where(vi > 0, np.append(0, y), np.append(y, 0))
    dy = F[1:] - F[:-1]
    return y - dy / a * dt / dx


def diffuse(y, d, dx, dt, ai=None):
    from scipy.linalg import solve_banded
    if ai is None:
        ai = a = 1
    else:
        a = (ai[1:] + ai[:-1]) / 2    
    q = d * dt / dx ** 2
    
    L = - q * ai[1:]
    U = - q * ai[:-1]
    A = a * (1 + q * 2) 
    B = a * y
    
    A[[0,-1]] = ai[[1,-2]] / 2 * (1 + q * 2) 
    return solve_banded((1, 1), [U, A, L], B, True, True)


In [29]:
R = 695e8
u = 800
d = 1e12

dx = dlat_1 * np.pi / 180 * R
dt = 24 * 3600

xi = np.arange(-90,90.5)
vi = u * np.sin(2 * xi * np.pi / 180)
li = 2 * np.pi * R * np.cos(xi * np.pi / 180)
y = flux_density_1.copy()

for _ in range(4 * 30):
    y = advect(y, vi, dx, dt, li)    
    #y = diffuse(y, d, dx, dt, li)


In [30]:
x = np.arange(-90,90) + 0.5

plt.figure(figsize=(10,8))
plt.plot(x, flux_density_1)
plt.plot(x, flux_density_2)
plt.plot(x, y)
plt.xlim(-90,90)
plt.ylim(-15,15)
plt.grid(True)
plt.tight_layout()